<a href="https://colab.research.google.com/github/kevin-blasiak-curtin/ISYS2001-Archive/blob/main/Module%2003%20-%20Making%20Computers%20Think/lab_ticket_budget_decisions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 Lab Ticket: Budget Decisions

This week your job is not to hand-code a program. It is to write **one complete, detailed prompt** that an AI can turn into a working budget tool in a single go.

In the worksheet you practised giving AI clear intent. Here that is the whole task. The skill being tested is whether you can describe exactly what you want, in enough detail, that one prompt produces the finished tool with no back-and-forth.

## What to submit

1. **Your final prompt**: one complete, self-contained prompt (paste it into the prompt cell below).
2. **The code it produced**: paste the AI-generated program into the code cell and run it, so you can show it works.

Your prompt can be the result of several rounds of refining. What you hand in is the single finished version: the one that would generate the whole tool if someone ran it cold.

## The brief

Design a tool that helps someone make a smart budget decision. What that means, and who it is for, is up to you. Some directions, though you are not limited to these:

- An expense classifier that sorts spending into categories and reacts to each.
- A budget checker that warns when an expense is too large a share of someone's budget.
- A savings goal tracker that gives different feedback depending on progress.
- A purchase advisor that weighs a price against how much money and income the person has.

Pick one or invent your own. The only hard requirement is that the finished program makes genuinely different decisions depending on the numbers it is given.

## How this works

You are going to write a prompt, hand it to an AI, and see what it builds. Your first attempt will not be right. Read what comes back and look for the gaps:

- Did it invent a rule you did not ask for? Your prompt was not precise enough.
- Did it miss a case? Say so explicitly next time.
- Did it ask you follow-up questions? A complete prompt should not need any.

Fold each fix back into the prompt and run it again. Keep going until one single prompt produces the whole working tool in one go. That final version is what you submit.

How detailed does it need to be? Detailed enough that a classmate could paste it cold and get the same working tool. Work out for yourself what that takes.

## Your final prompt

> Paste your complete, finished prompt below (double-click to edit).

```
(your prompt here)
```Build a complete, self-contained Savings Goal Tracker as a single Python script (standard library only — no external packages, no pip installs, no files written to or read from disk). It must run immediately with python3 savings_tracker.py in any terminal with no further edits, using an interactive command-line menu.

In [ ]:
#!/usr/bin/env python3
"""
Savings Goal Tracker
---------------------
A command-line savings goal tracker that gives different advice depending
on how your progress compares to where you "should" be on a linear plan.

No external dependencies. No files written to disk — everything is kept
in memory for the duration of the session, so it runs the same way every
time with python3 savings_tracker.py
"""

from datetime import date, timedelta


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def fmt_money(n):
    """Format a number as currency, rounding to whole dollars."""
    if n is None:
        return "$0"
    sign = "-" if n < 0 else ""
    return f"{sign}${abs(round(n)):,}"


def fmt_date(d):
    return d.strftime("%d %b %Y") if d else "—"


def parse_date(prompt_text, default=None):
    """Prompt for a date in YYYY-MM-DD format. Returns None on invalid input
    (after warning), or the default if the user presses Enter."""
    raw = input(prompt_text).strip()
    if not raw and default is not None:
        return default
    try:
        y, m, d = (int(x) for x in raw.split("-"))
        return date(y, m, d)
    except (ValueError, TypeError):
        print("  Couldn't read that date — use YYYY-MM-DD format.")
        return None


def parse_float(prompt_text, default=None, min_value=None):
    raw = input(prompt_text).strip()
    if not raw and default is not None:
        return default
    try:
        val = float(raw)
        if min_value is not None and val < min_value:
            print(f"  Value must be at least {min_value}.")
            return None
        return val
    except ValueError:
        print("  That's not a valid number.")
        return None


def clamp(n, lo, hi):
    return max(lo, min(hi, n))


# ---------------------------------------------------------------------
# Core data model
# ---------------------------------------------------------------------

class SavingsGoal:
    def __init__(self, name, target, current, start_date, target_date):
        self.name = name
        self.target = target
        self.current = current
        self.start_date = start_date
        self.target_date = target_date
        self.contributions = []  # list of (date, amount)

    def add_contribution(self, amount, on_date=None):
        on_date = on_date or date.today()
        self.contributions.append((on_date, amount))
        self.current += amount

    # -------------------------------------------------------------
    # Calculations
    # -------------------------------------------------------------

    def calculate(self):
        """Return a dict of all derived figures and the current status."""
        today = date.today()

        result = {
            "valid": True,
            "error": None,
            "goal_reached": self.current >= self.target,
        }

        if self.start_date > self.target_date:
            result["valid"] = False
            result["error"] = "Start date must be before the target date."
            return result

        total_days = max((self.target_date - self.start_date).days, 1)
        elapsed_days = clamp((today - self.start_date).days, 0, total_days)
        days_remaining = (self.target_date - today).days

        actual_pct = clamp((self.current / self.target) * 100, 0, 999) if self.target > 0 else 0
        expected_pct = clamp((elapsed_days / total_days) * 100, 0, 100)
        remaining = self.target - self.current
        overdue = (not result["goal_reached"]) and days_remaining < 0

        # --- status logic (mirrors the web version's thresholds) ---
        if result["goal_reached"]:
            status = "done"
        elif overdue:
            status = "critical"
        else:
            gap = actual_pct - expected_pct
            if gap > 10:
                status = "ahead"
            elif gap >= -10:
                status = "on_track"
            elif gap < -25 or (days_remaining < 30 and actual_pct < 90):
                status = "critical"
            else:
                status = "behind"

        # --- required contribution rate to hit the target date ---
        days_left_for_rate = max(days_remaining, 1)
        required_daily = remaining / days_left_for_rate if remaining > 0 else 0
        required_weekly = required_daily * 7
        required_monthly = required_daily * 30.44

        # --- projected completion date, based on actual contribution pace ---
        projected_date = None
        if self.contributions:
            total_contributed = sum(a for _, a in self.contributions)
            earliest = min(d for d, _ in self.contributions)
            span_days = max((today - earliest).days, 1)
            pace_daily = total_contributed / span_days
            if pace_daily > 0 and remaining > 0:
                days_to_goal = -(-int(remaining / pace_daily))  # ceil
                projected_date = today + timedelta(days=days_to_goal)

        result.update({
            "today": today,
            "total_days": total_days,
            "elapsed_days": elapsed_days,
            "days_remaining": days_remaining,
            "actual_pct": actual_pct,
            "expected_pct": expected_pct,
            "remaining": remaining,
            "status": status,
            "required_daily": required_daily,
            "required_weekly": required_weekly,
            "required_monthly": required_monthly,
            "projected_date": projected_date,
        })
        return result

    # -------------------------------------------------------------
    # Advice text, tuned to progress status
    # -------------------------------------------------------------

    def build_advice(self, r):
        if not r["valid"]:
            return "Almost ready", r["error"], []

        if r["goal_reached"]:
            title = "Goal reached — nice work"
            if r["days_remaining"] >= 0:
                body = (f"You've saved {fmt_money(self.current)} toward a "
                         f"{fmt_money(self.target)} target, {r['days_remaining']} "
                         f"day(s) ahead of your {fmt_date(self.target_date)} deadline.")
            else:
                body = (f"You've saved {fmt_money(self.current)} toward a "
                         f"{fmt_money(self.target)} target, after your original "
                         f"deadline of {fmt_date(self.target_date)}.")
            tips = [
                "Set a new, larger goal to keep the momentum going",
                "Move the surplus into a separate savings goal",
                "Automate this contribution amount going forward",
            ]
            return title, body, tips

        if r["status"] == "ahead":
            body = (f"You're at {r['actual_pct']:.0f}% saved versus an expected "
                     f"{r['expected_pct']:.0f}% for this point in your plan. "
                     f"{fmt_money(r['remaining'])} left to reach {fmt_money(self.target)}.")
            if r["projected_date"]:
                body += (f" At your current pace you're projected to finish by "
                          f"{fmt_date(r['projected_date'])}, ahead of your "
                          f"{fmt_date(self.target_date)} target.")
            tips = [
                "Increase your target amount since you have room to spare",
                "Redirect the surplus into a new goal",
                "Keep your current contribution rate — it's working",
            ]
            return "Ahead of schedule", body, tips

        if r["status"] == "on_track":
            body = (f"You're at {r['actual_pct']:.0f}% saved, close to the "
                     f"{r['expected_pct']:.0f}% expected by now. Keep contributing "
                     f"around {fmt_money(r['required_weekly'])} a week to reach "
                     f"{fmt_money(self.target)} by {fmt_date(self.target_date)}.")
            return "On track", body, []

        if r["status"] == "behind":
            body = (f"You're at {r['actual_pct']:.0f}% saved versus an expected "
                     f"{r['expected_pct']:.0f}%. To still hit {fmt_money(self.target)} "
                     f"by {fmt_date(self.target_date)}, you now need about "
                     f"{fmt_money(r['required_daily'])} a day "
                     f"({fmt_money(r['required_weekly'])} a week).")
            tips = [
                "A modest boost on top of your current rate would close the gap sooner",
                "Review recurring subscriptions or expenses for quick wins",
                "If that pace isn't realistic, consider extending your target date instead",
            ]
            return "Falling behind pace", body, tips

        # critical
        overdue_note = ""
        if r["days_remaining"] < 0:
            overdue_note = f" Your target date of {fmt_date(self.target_date)} has passed."
        days_note = f" with {r['days_remaining']} day(s) remaining." if r["days_remaining"] >= 0 else "."
        body = f"You have {fmt_money(r['remaining'])} left to save{days_note}{overdue_note}"
        tips = [
            f"Option A: contribute {fmt_money(r['required_daily'])} a day "
            f"({fmt_money(r['required_monthly'])}/month) to still hit {fmt_date(self.target_date)}",
            "Option B: keep your current pace and reset the target date to a more realistic one",
            "Log contributions weekly to see the change immediately",
        ]
        return "Off pace — needs a decision", body, tips


# ---------------------------------------------------------------------
# Display
# ---------------------------------------------------------------------

STATUS_LABELS = {
    "done": "GOAL REACHED",
    "ahead": "AHEAD OF SCHEDULE",
    "on_track": "ON TRACK",
    "behind": "BEHIND PACE",
    "critical": "CRITICAL",
}


def print_bar(pct, width=30):
    filled = int(clamp(pct, 0, 100) / 100 * width)
    return "[" + "#" * filled + "-" * (width - filled) + f"] {pct:.0f}%"


def print_status(goal):
    r = goal.calculate()
    print("\n" + "=" * 50)
    print(f"  {goal.name}")
    print("=" * 50)

    if not r["valid"]:
        print(f"  ! {r['error']}")
        print("=" * 50)
        return

    label = STATUS_LABELS[r["status"]]
    print(f"  Status: {label}")
    print(f"  Target:     {fmt_money(goal.target)}")
    print(f"  Saved:      {fmt_money(goal.current)}")
    print(f"  Remaining:  {fmt_money(max(r['remaining'], 0))}")
    days_txt = str(r["days_remaining"]) if r["days_remaining"] >= 0 else "Overdue"
    print(f"  Days left:  {days_txt}")
    print()
    print(f"  Progress: {print_bar(r['actual_pct'])}")
    print(f"  Expected: {print_bar(r['expected_pct'])}")
    print()

    title, body, tips = goal.build_advice(r)
    print(f"  --- {title} ---")
    print(f"  {body}")
    for tip in tips:
        print(f"    • {tip}")
    print("=" * 50)


def print_history(goal):
    print("\nContribution history:")
    if not goal.contributions:
        print("  No contributions logged yet.")
        return
    for d, amt in sorted(goal.contributions, key=lambda c: c[0], reverse=True):
        print(f"  {fmt_date(d)}   +{fmt_money(amt)}")


# ---------------------------------------------------------------------
# Setup + main loop
# ---------------------------------------------------------------------

def setup_goal():
    print("Set up your savings goal (press Enter to accept the default in brackets)\n")

    name = input("Goal name [New laptop]: ").strip() or "New laptop"

    target = None
    while target is None:
        target = parse_float("Target amount ($) [2500]: ", default=2500, min_value=0.01)

    current = None
    while current is None:
        current = parse_float("Current amount saved ($) [600]: ", default=600, min_value=0)

    today = date.today()
    start = None
    while start is None:
        start = parse_date(f"Start date (YYYY-MM-DD) [{today.isoformat()}]: ", default=today)

    default_target_date = start + timedelta(days=182)
    target_date = None
    while target_date is None:
        target_date = parse_date(
            f"Target date (YYYY-MM-DD) [{default_target_date.isoformat()}]: ",
            default=default_target_date,
        )

    return SavingsGoal(name, target, current, start, target_date)


def main():
    print("Savings Goal Tracker (Python CLI)")
    print("-" * 50)
    goal = setup_goal()
    print_status(goal)

    menu = """
What next?
  1) Log a contribution
  2) View status and advice
  3) View contribution history
  4) Edit goal (target amount, current saved, dates)
  5) Quit
"""
    while True:
        print(menu)
        choice = input("Choose 1-5: ").strip()

        if choice == "1":
            amt = None
            while amt is None:
                amt = parse_float("Contribution amount ($): ", min_value=0.01)
            goal.add_contribution(amt)
            print(f"  Logged {fmt_money(amt)}. New total: {fmt_money(goal.current)}")
            print_status(goal)

        elif choice == "2":
            print_status(goal)

        elif choice == "3":
            print_history(goal)

        elif choice == "4":
            new_target = parse_float(f"New target amount ($) [{goal.target}]: ", default=goal.target, min_value=0.01)
            if new_target is not None:
                goal.target = new_target
            new_current = parse_float(f"New current saved ($) [{goal.current}]: ", default=goal.current, min_value=0)
            if new_current is not None:
                goal.current = new_current
            new_target_date = parse_date(
                f"New target date (YYYY-MM-DD) [{goal.target_date.isoformat()}]: ",
                default=goal.target_date,
            )
            if new_target_date is not None:
                goal.target_date = new_target_date
            print_status(goal)

        elif choice == "5":
            print("Goodbye — keep saving!")
            break

        else:
            print("  Please enter a number from 1 to 5.")


if __name__ == "__main__":
    main()


Savings Goal Tracker (Python CLI)
--------------------------------------------------
Set up your savings goal (press Enter to accept the default in brackets)

Goal name [New laptop]: New Laptop
Target amount ($) [2500]: 2500
Current amount saved ($) [600]: 600
Start date (YYYY-MM-DD) [2026-08-10]: 10/8/2026
  Couldn't read that date — use YYYY-MM-DD format.
Start date (YYYY-MM-DD) [2026-08-10]: 2026-08-10
Target date (YYYY-MM-DD) [2027-02-08]: 2027-08-10

  New Laptop
  Status: AHEAD OF SCHEDULE
  Target:     $2,500
  Saved:      $600
  Remaining:  $1,900
  Days left:  365

  Progress: [#######-----------------------] 24%
  Expected: [------------------------------] 0%

  --- Ahead of schedule ---
  You're at 24% saved versus an expected 0% for this point in your plan. $1,900 left to reach $2,500.
    • Increase your target amount since you have room to spare
    • Redirect the surplus into a new goal
    • Keep your current contribution rate — it's working

What next?
  1) Log a contr

## Quick reflection

> Double-click to answer.

1. What did your first prompt miss that you had to add?
My first prompt missed stating it needed to be Python I had to change this and then. check if it worked as well as change the prompt.
2. Which single detail made the biggest difference to the output?
The. biggest difference was use strong descriptive words to make the prompt as short and presise as possible.
3. Could a classmate run your final prompt cold and get a working tool? How do you know?
Yes, I checked it by coping it into a AI chat to tested it in the chat.

## Before you submit

- [ x] Your final prompt is one complete block that needs no follow-up questions.
- [ x] The generated code runs top to bottom with no errors.
- [ x] The program makes different decisions for different inputs.
- [ x] You can explain what every part of your prompt is doing and why.
- [ x] You have downloaded the notebook and submitted it as your Week 3 lab ticket.

This is the first brick in a bigger wall. Over the semester your budget logic grows into a finance tracker, and the prompt-writing skill you practise here is one you will lean on the whole way.